In [2]:
from common.spark_session import spark, display_df, reconnect

In [16]:
sql = """
select *
from spark_catalog.marts.agg_customers
"""

spark.sql(sql).show()

+-----------+-------------+--------------------+--------+------------+-----+---------------+----------+-----------+
|customer_id|    full_name|               email|   phone|        city|state|membership_tier| joined_at|tenure_days|
+-----------+-------------+--------------------+--------+------------+-----+---------------+----------+-----------+
|       C017|Quinn Johnson|quinn.johnson@exa...|555-0117|Philadelphia|   PA|         bronze|2023-02-14|       1226|
|       C015|  Olivia Chen|olivia.chen@examp...|555-0115| Minneapolis|   MN|           gold|2020-03-03|       2304|
|       C011| Karen Taylor|karen.taylor@exam...|555-0111|     Atlanta|   GA|       platinum|2018-05-12|       2965|
|       C002|    Bob Smith|bob.smith@example...|555-0102|    New York|   NY|         silver|2021-06-20|       1830|
+-----------+-------------+--------------------+--------+------------+-----+---------------+----------+-----------+



In [15]:
sql = """
select *
from iceberg_catalog.marts.agg_customers_iceberg
"""

spark.sql(sql).show()

+-----------+-------------+--------------------+--------+------------+-----+---------------+----------+-----------+
|customer_id|    full_name|               email|   phone|        city|state|membership_tier| joined_at|tenure_days|
+-----------+-------------+--------------------+--------+------------+-----+---------------+----------+-----------+
|       C017|Quinn Johnson|quinn.johnson@exa...|555-0117|Philadelphia|   PA|         bronze|2023-02-14|       1226|
|       C015|  Olivia Chen|olivia.chen@examp...|555-0115| Minneapolis|   MN|           gold|2020-03-03|       2304|
|       C011| Karen Taylor|karen.taylor@exam...|555-0111|     Atlanta|   GA|       platinum|2018-05-12|       2965|
|       C002|    Bob Smith|bob.smith@example...|555-0102|    New York|   NY|         silver|2021-06-20|       1830|
+-----------+-------------+--------------------+--------+------------+-----+---------------+----------+-----------+



In [31]:
sql = """
WITH src AS (
  SELECT
    *
  FROM spark_catalog.staging.stg_orders_quality
), 
valid_customers AS (
  SELECT
    customer_id
  FROM spark_catalog.staging.stg_customers
)
SELECT
  s.*,
  CASE
    WHEN s.amount <= 0
    THEN 'non_positive_amount'
    WHEN NOT s.status IN ('completed', 'refunded', 'pending')
    THEN 'invalid_status'
    WHEN s.customer_id NOT IN (
      SELECT
        customer_id
      FROM valid_customers
    )
    THEN 'orphan_customer'
  END AS quarantine_reason
FROM src AS s
WHERE
  s.amount <= 0
  OR NOT s.status IN ('completed', 'refunded', 'pending')
  OR s.customer_id NOT IN (
    SELECT
      customer_id
    FROM valid_customers
  )
"""
spark = reconnect()
spark.sql(sql).limit(20).show(truncate=False)

Reconnected: fresh Spark session created.
+--------+-----------+------+---------+-------------------+-------------------+
|order_id|customer_id|amount|status   |ordered_at         |quarantine_reason  |
+--------+-----------+------+---------+-------------------+-------------------+
|2001    |C002       |-9.99 |completed|2024-03-05 11:00:00|non_positive_amount|
|2002    |C999       |30.0  |completed|2024-03-05 12:00:00|orphan_customer    |
|2003    |C003       |20.0  |shipped  |2024-03-05 13:00:00|invalid_status     |
+--------+-----------+------+---------+-------------------+-------------------+

